# Watt The Hack - Local Playtester Sandbox (v0.1.0)

Welcome to the playtester sandbox! This notebook allows you to test controllers against hidden scenarios *entirely on your local machine*.
You don't need to ZIP files or wait for cloud evaluation. It runs instantly and prints the exact same metrics and cost breakdown the cloud judge uses.

## 1. Install the Engine & Fetch Scenarios

Run this cell to download the engine and all the scenario files directly from the public GitHub repository into your Colab environment.

In [ ]:
import os

# If you haven't cloned the repo, we'll clone it now to get the `scenarios/` folder!
if not os.path.exists("Watt-The-Hack-Engine-Public"):
    !git clone https://github.com/AaronEliasZachariah/Watt-The-Hack-Engine-Public.git
    
# Change working directory so python can find the scenarios/ folder
if os.path.exists("Watt-The-Hack-Engine-Public"):
    os.chdir("Watt-The-Hack-Engine-Public")

# Install the engine into the python environment
!pip install -e .

## 2. Write your Agent

Here is your controller. This exact class shape is identical to what you will eventually submit to the cloud. You can use OpenAI calls inside the `.plan()` method to make strategic decisions based on the scenario briefing.

In [ ]:
import os
# import openai

class Strategy:
    def __init__(self):
        # Local evaluation allows you to use your real API keys!
        self.api_key = os.environ.get("OPENAI_API_KEY", "dummy-key")
        self.target_soc = 0.5
        
    def plan(self, state: dict) -> dict:
        """
        Called ONCE at the start. 
        Read the briefing, make an LLM call, and decide your strategy!
        """
        briefing = state.get("qualitative_briefing", "")
        print(f"\n[Agent] Reading briefing: {briefing}")
        
        # Example strategy selection:
        self.target_soc = 0.8
        print(f"[Agent] Decided on target SOC: {self.target_soc}\n")
        
        return {"agent_plan": {"target_soc": self.target_soc}}

    def replan(self, state: dict, alerts: list) -> dict:
        """
        Called mid-run if a 'qualitative_alert' event fires (e.g. weather changing).
        """
        print(f"\n[Agent] ALERT! {alerts}")
        self.target_soc = 0.2
        return {"agent_plan": {"target_soc": self.target_soc}}

    def step(self, state: dict) -> dict:
        """
        Called every 15-minute timestep. 
        NO LLM calls here, just execute the plan quickly!
        """
        plan = state.get("agent_plan", {})
        target = plan.get("target_soc", self.target_soc)
        current_soc = state.get("soc", 0.0)
        
        flow = 0.0
        if current_soc < target - 0.05:
            flow = -50.0  # charge battery
        elif current_soc > target + 0.05:
            flow = 50.0   # discharge battery
            
        return {
            "battery_flow_kw": flow,
            "emergency_generator": 0.0,
            "curtail_solar": 0.0,
            "fcas_reserve_kw": 0.0
        }

## 3. Run the Backtest Locally

Pick a `scenario_id` to evaluate against. This code mimics exactly what the cloud evaluator does behind the scenes, including hooking up `.plan()` and tracking your detailed cost breakdown.

In [ ]:
import json
from watt_the_hack.simulation.runner import run_simulation
from watt_the_hack.data_loaders.scenarios import load_scenario, find_scenario_by_id

# ── PICK YOUR SCENARIO ─────────────────────────
SCENARIO_ID = "duck_curve"  # change this to test other scenarios!
# ───────────────────────────────────────────────

print(f"Loading scenario: {SCENARIO_ID}")
scenario_path = find_scenario_by_id(SCENARIO_ID)
if not scenario_path:
    raise FileNotFoundError(f"Could not find scenario {SCENARIO_ID} in the scenarios/ folder!")

spec, initial_state = load_scenario(scenario_path)
strategy = Strategy()

# 1. Trigger the one-shot plan
if hasattr(strategy, "plan"):
    plan_result = strategy.plan(initial_state)
    if isinstance(plan_result, dict):
        initial_state.update(plan_result)

# 2. Wrap the controller to handle mid-run replans
def controller_wrapper(state):
    current_t = int(state.get("time", 0))
    alerts = [
        e for e in state.get("events", [])
        if e.get("type") == "qualitative_alert" and int(e.get("at_step", -1)) == current_t
    ]
    if alerts and hasattr(strategy, "replan"):
        update = strategy.replan(state, alerts)
        if isinstance(update, dict):
            state["agent_plan"] = {**state.get("agent_plan", {}), **update.get("agent_plan", {})}
    return strategy.step(state)

# 3. Run the Simulation Loop!
print("Evaluating... (this should be instant!)")
result = run_simulation(
    controller=controller_wrapper, 
    initial_state=initial_state,
    steps=len(initial_state.get("profiles", {}).get("demand", []))
)

# 4. Aggregate and display exactly like the cloud
all_breakdowns = [out.get("cost_breakdown", {}) for out in result["outputs"]]
agg_breakdown = {}
for bd in all_breakdowns:
    for k, v in bd.items():
        agg_breakdown[k] = agg_breakdown.get(k, 0.0) + float(v)

print("\n✅ Evaluation Complete!")
print(f"\nFinal Score (Cost): ${result['metrics']['final_score']:.2f}")
print("\n--- Cost Breakdown ---")
print(json.dumps(agg_breakdown, indent=2))
